In [ ]:
!pip install mediapipe==0.10.14 opencv-python --quiet

In [46]:
import cv2
import gradio as gr
import mediapipe as mp
import numpy as np
from tqdm import tqdm

In [40]:
def mediapipe_markdown(video_path: str) -> list:
    mp_hands = mp.solutions.hands
    mp_draw = mp.solutions.drawing_utils
    
    hands = mp_hands.Hands(
        static_image_mode=False,
        max_num_hands=2,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    )
    
    cap = cv2.VideoCapture(video_path)
    if cap.isOpened():
    
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = cap.get(cv2.CAP_PROP_FPS)
        
        out = cv2.VideoWriter(
            "output.mp4",
            cv2.VideoWriter_fourcc(*'mp4v'),
            fps,
            (width, height)
        )
        dots, frames, video = [], [], []
        while cap.isOpened():
            success, frame = cap.read()
            if not success:
                break
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = hands.process(rgb_frame)
            if results.multi_hand_landmarks:
                for hand_landmarks in results.multi_hand_landmarks:
                    mp_draw.draw_landmarks(
                        frame,
                        hand_landmarks,
                        mp_hands.HAND_CONNECTIONS,
                        mp_draw.DrawingSpec(color=(0, 255, 0), thickness=2, circle_radius=3),
                        mp_draw.DrawingSpec(color=(255, 0, 0), thickness=2)
                    )
            out.write(frame)
            
            if results.multi_hand_landmarks:
                for hand_landmarks in results.multi_hand_landmarks:
                    for index, coord in enumerate(hand_landmarks.landmark):
                        dots.append(coord.x)
                        dots.append(coord.y)
                        dots.append(coord.z)
                        frames.append(dots)
                        dots = []
                    video.append(frames)
                    frames = []
        cap.release()
        out.release()
        cv2.destroyAllWindows()
        return video
    else:
        return "Video was not opened"


In [41]:
def video_preprocessing(video_coords: list) -> list:
    video_preprocessed = []
    video_np = np.array(video)
    A, B, C = video_np.shape
    old_time = np.linspace(0, 1, A)
    new_time = np.linspace(0, 1, 48)
    resampled_video = np.zeros((48, 21, 3), dtype=video_np.dtype)
    for b in range(B):
        for c in range(C):
            resampled_video[:, b, c] = np.interp(
                new_time,
                old_time,
                video_np[:, b, c]
            )
    wrist = resampled_video[:, 0:1, :]
    resampled_video_normalized = resampled_video-wrist
    resampled_video_normalized = resampled_video_normalized.reshape(48, 63)
    video_preprocessed.append(resampled_video_normalized)
    return video_preprocessed

In [42]:
video_path = "/kaggle/input/datasets/arseniipolyakov/video-example/document_5213285575091593250.mp4"
video_coords = mediapipe_markdown(video_path=video_path)
video_coords_preprocessed = video_preprocessing(video_coords=video_coords)

W0000 00:00:1779111053.668422    2103 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1779111053.711876    2103 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


(1, 48, 63)


In [45]:
print(video_coords_preprocessed[0])

[[ 0.          0.          0.         ...  0.0606119  -0.21215975
   0.00641495]
 [ 0.          0.          0.         ...  0.05782301 -0.24939844
   0.01912733]
 [ 0.          0.          0.         ...  0.07523864 -0.26263827
   0.05221723]
 ...
 [ 0.          0.          0.         ... -0.16755223 -0.42789033
   0.31821123]
 [ 0.          0.          0.         ... -0.17479616 -0.41518007
   0.33092456]
 [ 0.          0.          0.         ... -0.06039834 -0.37642443
   0.22532197]]


In [ ]:
demo = gr.Interface(
    fn=classify_video,
    inputs=gr.Video(label="Upload MP4"),
    outputs=gr.Label(label="Prediction"),
    title="Video Classification",
    description="Upload mp4 video and get class prediction"
)

demo.launch()